# 03 — Silver RENAMU
## Transformación Bronze → Silver (Modelo EAV)

Procesa el **Registro Nacional de Municipalidades (RENAMU)** aplicando la estrategia **"Embudo EAV"**.

**El problema a resolver**: Entre 2012 y 2020, cada ZIP del RENAMU contiene decenas de módulos CSV (uno por formulario). A partir de 2021, es un único CSV plano. Si uniéramos los años directamente, el DataFrame de Silver tendría miles de columnas dispersas y llenas de nulos.

**La solución**: En lugar de unir tablas anchas, rotamos (Unpivot) cada archivo CSV módulo a módulo hacia el modelo **Entidad-Atributo-Valor (EAV)** antes de consolidar. El resultado siempre tiene el mismo esquema estrecho: `UBIGEO | ANO_RENAMU | MODULO | PREGUNTA | RESPUESTA`.

**Fuente:** `data/bronze/renamu/{año}/batch_*.json` (con columna `_source_file` por módulo)  
**Destino:** `data/silver/renamu/` (Parquet EAV particionado por ANO_RENAMU)

In [9]:
import sys, time
from pathlib import Path
from functools import reduce

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from src.paths import BRONZE, SILVER, CATEGORIAS_CSV, str_path, ensure_dirs
from src.spark_utils import get_spark
from src.transformations.common import clean_ghost_nulls
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

Imports OK


In [10]:
# ── Configuración ─────────────────────────────────────────────────────────────
BRONZE_BASE = BRONZE["renamu"]
SILVER_PATH = str_path(SILVER["renamu"])

# Columnas que identifican a la municipalidad (no son preguntas del cuestionario)
# Estas columnas se extraerán de cada módulo y se mantienen como dimensiones.
# Las búsqueda es case-insensitive.
ID_COL_CANDIDATES = [
    "ubigeo", "ccdd", "ccpp", "ccdi", "idmunici",
    "departamento", "provincia", "distrito",
    "ano_renamu", "_source_file",
]

# Tamaño de lote para el Unpivot (columnas por iteración de STACK)
# Reducir si se producen errores de memoria
CHUNK_SIZE = 80

print(f"Bronze source : {BRONZE_BASE}")
print(f"Silver target : {SILVER_PATH}")
ensure_dirs()

Bronze source : /workspace/data/bronze/renamu
Silver target : /workspace/data/silver/renamu
[OK] Estructura de directorios verificada en /workspace/data


In [11]:
spark = get_spark(app_name="MEF_Silver_RENAMU", memory="4g")
spark.conf.set("spark.sql.caseSensitive", "true")

[OK] SparkSession lista | version=3.5.0 | master=local[*]


## PARTE 1: Lectura Bronze y Detección de Módulos

Los JSONs de Bronze contienen la columna `_source_file` que indica de qué módulo/CSV vino cada registro. Esto permite procesar cada módulo de forma independiente, evitando la explosión de columnas.

In [12]:
if not BRONZE_BASE.exists() or not any(BRONZE_BASE.rglob("*.json")):
    raise FileNotFoundError(
        f"Sin datos Bronze RENAMU en {BRONZE_BASE}.\n"
        "Ejecutar primero: notebooks/00_bronze_ingestion.ipynb"
    )

# Leer todos los años disponibles
dfs_by_year = []
for year_dir in sorted(d for d in BRONZE_BASE.iterdir() if d.is_dir()):
    json_files = list(year_dir.rglob("*.json"))
    if not json_files:
        continue
    year_df = (
        spark.read.option("multiline", "false").json(str_path(year_dir))
        .withColumn("ANO_RENAMU", F.lit(year_dir.name))
    )
    dfs_by_year.append((year_dir.name, year_df))
    print(f"  Ano {year_dir.name}: {len(json_files)} archivo(s) JSON")

if not dfs_by_year:
    raise ValueError("No se encontraron datos JSON de RENAMU en Bronze.")

print(f"\nTotal años cargados: {len(dfs_by_year)}")

  Ano 2012: 110 archivo(s) JSON
  Ano 2013: 109 archivo(s) JSON
  Ano 2014: 98 archivo(s) JSON
  Ano 2015: 115 archivo(s) JSON
  Ano 2016: 111 archivo(s) JSON
  Ano 2017: 122 archivo(s) JSON
  Ano 2018: 47 archivo(s) JSON
  Ano 2019: 20 archivo(s) JSON
  Ano 2020: 19 archivo(s) JSON
  Ano 2021: 1 archivo(s) JSON
  Ano 2022: 1 archivo(s) JSON
  Ano 2023: 1 archivo(s) JSON
  Ano 2024: 1 archivo(s) JSON

Total años cargados: 13


## PARTE 2: Función de Unpivot EAV por Módulo

Para cada año y cada módulo detectado (a través de `_source_file`), aplicamos:
1. Limpieza de ghost nulls
2. Reconstrucción de UBIGEO de 6 dígitos
3. Unpivot (STACK) de las columnas-pregunta en filas `PREGUNTA | RESPUESTA`

El resultado unificado siempre tiene el mismo esquema, sin importar el año.

In [13]:
def detect_id_cols(df):
    """Detecta las columnas de identificación presentes en el DataFrame (case-insensitive)."""
    return [c for c in df.columns if c.lower() in ID_COL_CANDIDATES]

def build_ubigeo(df):
    cols_lower = {c.lower(): c for c in df.columns}

    def clean_id(col_ref):
        c_str = F.col(col_ref).cast("string")
        # 1. Elimina el sufijo decimal .0 si se interpretó como float
        c_no_dec = F.regexp_replace(c_str, r"\.0$", "")
        # 2. Elimina espacios en blanco estándar y espacios de no-ruptura (\xa0)
        c_no_spaces = F.regexp_replace(c_no_dec, r"[\s\xa0]+", "")
        return c_no_spaces

    # 1. Intentar con 'ubigeo'
    if "ubigeo" in cols_lower:
        return df.withColumn("UBIGEO", F.lpad(clean_id(cols_lower["ubigeo"]), 6, "0"))

    # 2. Intentar con 'idmunici'
    if "idmunici" in cols_lower:
        return df.withColumn("UBIGEO", F.lpad(clean_id(cols_lower["idmunici"]), 6, "0"))

    # 3. Concatenar CCDD + CCPP + CCDI
    if all(k in cols_lower for k in ["ccdd", "ccpp", "ccdi"]):
        return df.withColumn("UBIGEO", F.concat(
            F.lpad(clean_id(cols_lower["ccdd"]), 2, "0"),
            F.lpad(clean_id(cols_lower["ccpp"]), 2, "0"),
            F.lpad(clean_id(cols_lower["ccdi"]), 2, "0"),
        ))

    return df.withColumn("UBIGEO", F.lit("000000"))

def unpivot_module(df, id_cols, question_cols, modulo_name):
    """
    Aplica el unpivot EAV a un subconjunto de columnas (un módulo),
    procesando en lotes de CHUNK_SIZE para evitar OutOfMemoryError.
    Retorna un DataFrame con esquema: UBIGEO | ANO_RENAMU | MODULO | PREGUNTA | RESPUESTA
    """
    cast_exprs = [F.col(c).cast(StringType()).alias(c) if c in question_cols else F.col(c) for c in df.columns]
    df = df.select(*cast_exprs)

    result_chunks = []
    for i in range(0, len(question_cols), CHUNK_SIZE):
        chunk = question_cols[i:i + CHUNK_SIZE]
        n = len(chunk)
        pairs = ", ".join([f"'{c}', `{c}`" for c in chunk])
        stack_expr = f"STACK({n}, {pairs}) AS (PREGUNTA, RESPUESTA)"

        id_exprs = [F.col(c) for c in id_cols]
        chunk_df = (
            df.select(*id_exprs, F.expr(stack_expr))
            .filter(F.col("RESPUESTA").isNotNull() & (F.col("RESPUESTA") != ""))
            .withColumn("MODULO", F.lit(modulo_name))
        )
        result_chunks.append(chunk_df)

    if not result_chunks:
        return None

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), result_chunks)

print(f"Funciones EAV definidas. Chunk size: {CHUNK_SIZE} columnas/lote.")


Funciones EAV definidas. Chunk size: 80 columnas/lote.


## PARTE 3: Procesamiento Año por Año → Módulo por Módulo

Para cada año y cada módulo (identificado por `_source_file`), se aplica el Unpivot EAV. Los resultados se apilan verticamente. Esto garantiza uniformidad del esquema entre 2012 y 2024.

In [14]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

control = ControlManager(pipeline_name="silver_renamu")
execution = control.start(input_parameters={"source": "renamu", "estrategia": "EAV"})

start_time = time.time()
total_filas = 0

try:
    for year_dir in sorted(d for d in BRONZE_BASE.iterdir() if d.is_dir()):
        year_str = year_dir.name
            
        json_files = list(year_dir.rglob("*.json"))
        if not json_files:
            continue
            
        print(f"\\n--- Procesando ano {year_str} ({len(json_files)} archivos) ---")
        year_eav = []
        
        for json_path in json_files:
            # 1. Lectura robusta forzando UTF-8
            file_df = (
                spark.read
                .option("multiline", "false")
                .option("encoding", "UTF-8")
                .json(str_path(json_path))
            )
            
            # 2. Limpieza dinámica del BOM (ï»¿) que corrompe nombres de columnas
            for c in file_df.columns:
                if 'ï»¿' in c:
                    file_df = file_df.withColumnRenamed(c, c.replace('ï»¿', ''))

            file_df = file_df.withColumn("ANO_RENAMU", F.lit(year_str))
            
            # 3. Transformaciones base
            clean_df = clean_ghost_nulls(file_df)
            clean_df = build_ubigeo(clean_df)
            
            # 4. Enrutamiento con EXCLUSIÓN ESTRICTA DE DIMENSIONES (Históricas y Actuales)
            DIM_COLS_TO_EXCLUDE = {
                # Columnas descriptivas antiguas
                "catmuni", "nom_prov", "nom_dist", "nom_dpto",
                # Columnas descriptivas nuevas
                "tipomuni", "departamento", "provincia", "distrito", "region", 
                # Códigos y variaciones de tiempo
                "ubigeo", "ccdd", "ccpp", "ccdi", "idmunici",
                "año", "ano", "aã±o", "ano_renamu", "_source_file"
            }
            
            id_cols = ["UBIGEO", "ANO_RENAMU"]
            
            # Filtro inteligente: Lo que no es ID y no está en la lista negra, es pregunta.
            question_cols = [
                c for c in clean_df.columns 
                if c.lower() not in DIM_COLS_TO_EXCLUDE 
                and c.lower() not in ID_COL_CANDIDATES
                and c not in id_cols
            ]
            
            # Determinar si es modular (tiene _source_file) o es el plano único anual
            modulo_name = "UNICO"
            if "_source_file" in [c.lower() for c in clean_df.columns]:
                modulo_name = clean_df.select("_source_file").first()[0]
                
            if question_cols:
                eav_df = unpivot_module(clean_df, id_cols, question_cols, modulo_name)
                if eav_df:
                    year_eav.append(eav_df)

        if not year_eav:
            print(f"  [!] No se generaron datos para el ano {year_str}")
            continue

        print(f"  Consolidando archivos rotados de este ano...")
        year_final_df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), year_eav)

        # 5. Cruce con Catálogo de Categorías (si existe) y DQ Flag
        if CATEGORIAS_CSV.exists():
            cat_df = (
                spark.read.option("header", "true").csv(str_path(CATEGORIAS_CSV))
                .withColumn("UBIGEO_REF", F.lpad(F.col("UBIGEO").cast("string"), 6, "0"))
                .select("UBIGEO_REF", "CATEGORIA")
                .dropDuplicates(["UBIGEO_REF"])
            )
            year_final_df = (
                year_final_df.join(cat_df, year_final_df.UBIGEO == cat_df.UBIGEO_REF, "left")
                .drop("UBIGEO_REF")
            )
            
            year_final_df = year_final_df.withColumn(
                "DQ_MUNI_SIN_CATEGORIA",
                F.when(F.col("CATEGORIA").isNull(), F.lit(1).cast("byte")).otherwise(F.lit(0).cast("byte"))
            ).fillna({"CATEGORIA": "SIN_CATEGORIA"})

        print(f"  Escribiendo particion del ano {year_str} a disco...")
        (
            year_final_df.write
            .mode("overwrite")
            .partitionBy("ANO_RENAMU")
            .format("parquet")
            .save(SILVER_PATH)
        )
        
        n_year = year_final_df.count()
        total_filas += n_year
        print(f"  => {n_year:,} filas escritas para {year_str}.")

    elapsed = time.time() - start_time
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={"total_filas_eav": total_filas, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\\nSilver RENAMU completado en {elapsed:.1f}s")
    print(f"   Total filas EAV generadas: {total_filas:,}")

except Exception as e:
    control.log_error("SilverRenamuError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

2026-06-19 15:46:16 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución iniciada | pipeline=silver_renamu id=323e95da
\n--- Procesando ano 2012 (110 archivos) ---
  Consolidando archivos rotados de este ano...
  Escribiendo particion del ano 2012 a disco...
  => 2,579,381 filas escritas para 2012.
\n--- Procesando ano 2013 (109 archivos) ---
  Consolidando archivos rotados de este ano...
  Escribiendo particion del ano 2013 a disco...
  => 2,478,167 filas escritas para 2013.
\n--- Procesando ano 2014 (98 archivos) ---
  Consolidando archivos rotados de este ano...
  Escribiendo particion del ano 2014 a disco...
  => 2,159,390 filas escritas para 2014.
\n--- Procesando ano 2015 (115 archivos) ---
  Consolidando archivos rotados de este ano...
  Escribiendo particion del ano 2015 a disco...
  => 3,191,179 filas escritas para 2015.
\n--- Procesando ano 2016 (111 archivos) ---
  Consolidando archivos rotados de este ano...
  Escribiendo particion del ano 2016 a disco...
  => 2,99

In [15]:
# ── Verificación Final ────────────────────────────────────────────────────────
df_check = spark.read.parquet(SILVER_PATH)
print(f"Verificacion Silver RENAMU (EAV):")
print(f"  Total filas    : {df_check.count():,}")
print(f"  Columnas       : {df_check.columns}")
print(f"  Anos presentes :")
df_check.groupBy("ANO_RENAMU").count().orderBy("ANO_RENAMU").show(20)
print(f"  Top 10 preguntas mas frecuentes:")
df_check.groupBy("PREGUNTA").count().orderBy("count", ascending=False).show(10, truncate=False)

Verificacion Silver RENAMU (EAV):
  Total filas    : 29,450,221
  Columnas       : ['UBIGEO', 'PREGUNTA', 'RESPUESTA', 'MODULO', 'CATEGORIA', 'DQ_MUNI_SIN_CATEGORIA', 'ANO_RENAMU']
  Anos presentes :
+----------+-------+
|ANO_RENAMU|  count|
+----------+-------+
|      2012|2579381|
|      2013|2478167|
|      2014|2159390|
|      2015|3191179|
|      2016|2997097|
|      2017|3260087|
|      2018|2553848|
|      2019|1755276|
|      2020|1693708|
|      2021|1668832|
|      2022|1706815|
|      2023|1651862|
|      2024|1754579|
+----------+-------+

  Top 10 preguntas mas frecuentes:
+--------+-----+
|PREGUNTA|count|
+--------+-----+
|P57     |33312|
|VFI_P38 |32248|
|P45     |27715|
|p51     |27570|
|P83_02  |25813|
|P83_03  |25813|
|P83_01  |25631|
|P38_07  |24720|
|P38_01  |24720|
|P38_04  |24720|
+--------+-----+
only showing top 10 rows

